In [186]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [187]:
# Carregar os dados
df = pd.read_excel('../dataset/dataset_velocidade_v2.xlsx')

In [188]:
# Criar feature categórica baseada nos dois primeiros dígitos da velocidade
df['vel_str'] = df['velocidade'].apply(lambda x: str(x).split('.')[1][:2])  # '0.0813' -> '08'
# Criar target
df['target'] = (df['vel_str'] != '08').astype(int)  # 0 = normal, 1 = anomalia

In [189]:
df.head()

,movimento,tempo,velocidade,vel_str,target
0,avanco,1.230,0.081301,08,0
1,avanco,1.214,0.082372,08,0
2,avanco,1.213,0.082440,08,0
3,avanco,1.275,0.078431,07,1
4,avanco,1.405,0.071174,07,1


In [190]:
# Features e target
X = df[['movimento', 'tempo', 'vel_str']]
y = df['target']

# Colunas categóricas e numéricas
cat_features = ['movimento', 'vel_str']
num_features = ['tempo']


In [191]:
from sklearn.preprocessing import OneHotEncoder

# Pré-processamento atualizado
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)  # <--- handle_unknown
    ]
)

# Pipeline com Regressão Logística
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])


In [192]:
# Dividir dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [193]:
# Treinar modelo
pipeline.fit(X_train, y_train)

# Avaliar
y_pred = pipeline.predict(X_test)
print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred))
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred))

Matriz de Confusão:
[[262   0]
 [  0 576]]

Relatório de Classificação:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       262
           1       1.00      1.00      1.00       576

    accuracy                           1.00       838
   macro avg       1.00      1.00      1.00       838
weighted avg       1.00      1.00      1.00       838



In [202]:
import numpy as np

def testar_velocidade_random():
    velocidade_rand = np.round(np.random.uniform(0, 0.2), 5)
    vel_str = str(velocidade_rand).split('.')[1][:2]
    teste = pd.DataFrame({
        'movimento': ['avanco'],  # valor fictício
        'tempo': [1.0],
        'vel_str': [vel_str]
    })
    resultado = pipeline.predict(teste)[0]
    status = "Normal" if resultado == 0 else "Anomalia"
    print(f"Velocidade gerada: {velocidade_rand} → {status} -> {resultado}")

# Testar 10 velocidades aleatórias
for _ in range(10):
    testar_velocidade_random()


Velocidade gerada: 0.05778 → Anomalia -> 1
Velocidade gerada: 0.12654 → Anomalia -> 1
Velocidade gerada: 0.13279 → Anomalia -> 1
Velocidade gerada: 0.12197 → Anomalia -> 1
Velocidade gerada: 0.19198 → Anomalia -> 1
Velocidade gerada: 0.02657 → Anomalia -> 1
Velocidade gerada: 0.18132 → Anomalia -> 1
Velocidade gerada: 0.17538 → Anomalia -> 1
Velocidade gerada: 0.08904 → Normal -> 0
Velocidade gerada: 0.1031 → Anomalia -> 1


In [195]:
# def testar_velocidade_random():
#     import numpy as np
#     velocidade_rand = np.round(np.random.uniform(0, 0.2), 5)
    
#     # Extrair apenas os dois primeiros dígitos após o ponto decimal
#     decimal_str = str(velocidade_rand).split('.')[1][:2]  # '0.08794' -> '08'
#     status = "Normal" if decimal_str == "08" else "Anomalia"
    
#     print(f"Velocidade gerada: {velocidade_rand} → {status}")

# # Testar várias vezes
# for _ in range(10):
#     testar_velocidade_random()
